# Industry Data Cleaning

## Load data

In [1]:
import json
import unicodedata
from pathlib import Path

import pandas as pd

RAW_PATH = Path("../../../CalidadAireIndustrias_2025/emission_permits_anom_2.json")
OUT_DIR = Path("../Data")

with open(RAW_PATH, encoding="utf-8") as f:
    raw = json.load(f)

print("Features:", len(raw["features"]))
print("Feature types:", {f["type"] for f in raw["features"]})
print("Geometry types:", {f["geometry"]["type"] for f in raw["features"]})

Features: 544
Feature types: {'Feature'}
Geometry types: {'Point'}


## Flatten features

In [2]:
# Keep properties + coordinates, drop 'type'
df = pd.DataFrame([
    {**f["properties"],
     "Longitude": f["geometry"]["coordinates"][0],
     "Latitude": f["geometry"]["coordinates"][1]}
    for f in raw["features"]
])

df["IDExpediente"] = df["IDExpediente"].astype(int)
df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Cuenca,Longitude,Latitude
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,Río Bogotá,-74.226561,4.703418
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,Resguardo,None,Carbón,Caldera Horno,Río Suárez,-73.704281,5.318407
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera Horno,Río Bogotá,-74.210355,4.800462
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,Balsillas,None,Fuel Oil No.8,Planta de Asfalto,Río Bogotá,-74.284112,4.678797
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,El Hato,None,Carbón,Caldera Horno,Río Bogotá,-74.193752,4.699590


## Clean text values

In [3]:
text_cols = df.select_dtypes("object").columns

# Trim extra spaces
df[text_cols] = df[text_cols].apply(lambda s: s.str.strip().str.replace(r"\s+", " ", regex=True))

# Placeholder values treated as null
df[text_cols] = df[text_cols].replace({"(sin definir)": None, "": None})

In [4]:
# Standardize urban area names in Vereda
urban = {"casco urbano", "area urbana", "urbano", "centro urbano"}
df.loc[df["Vereda"].str.lower().isin(urban), "Vereda"] = "Casco Urbano"

df["Vereda"].value_counts().head(5)

Vereda
Casco Urbano     79
MOCHUELO BAJO    47
Patio Bonito     30
Pueblo Viejo     22
CENTRO           14
Name: count, dtype: int64

## Class values

In [5]:
df["Class"].value_counts(dropna=False)

Class
None                                                                                     223
Por emisiones atmosféricas sin cumplir con los requisitos de ley                         142
Por emisiones atmosféricas sin permiso o no cumplimiento con los términos del permiso    106
Emitir por encima de los parámetros establecidos en la norma                              73
Name: count, dtype: int64

In [6]:
# Class vs Estado
pd.crosstab(df["Estado"], df["Class"].fillna("NULL"))

Class,Emitir por encima de los parámetros establecidos en la norma,NULL,Por emisiones atmosféricas sin cumplir con los requisitos de ley,Por emisiones atmosféricas sin permiso o no cumplimiento con los términos del permiso
Estado,,,,
En Trámite,0,7,0,0
Sancionatorio,73,0,142,106
Seguimiento y Control,0,216,0,0


## Duplicates

In [7]:
print("Exact duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# Same IDExpediente with different location = different emission source
print("Repeated IDExpediente:", df["IDExpediente"].duplicated().sum())
print("Rows:", len(df))

Exact duplicates: 8
Repeated IDExpediente: 34
Rows: 536


## Industry ID

In [8]:
def slug(text):
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode()
    return "_".join(text.lower().replace("-", " ").split())

base = df["Municipio"].map(slug) + "-" + df["Vereda"].map(slug)

# Add a number only when the base id repeats
counts = base.map(base.value_counts())
seq = base.groupby(base).cumcount() + 1
df["id"] = base.where(counts == 1, base + "-" + seq.astype(str))

assert df["id"].is_unique
df = df[["id"] + [c for c in df.columns if c != "id"]]
df[["id", "Municipio", "Vereda"]].head(10)

,id,Municipio,Vereda
0,mosquera-centro,MOSQUERA,CENTRO
1,lenguazaque-resguardo-1,LENGUAZAQUE,Resguardo
2,madrid-la_punta,MADRID,LA PUNTA
3,mosquera-balsillas-1,MOSQUERA,Balsillas
4,funza-el_hato,FUNZA,El Hato
5,tausa-rasgata-1,TAUSA,Rasgatá
6,funza-la_isla-1,FUNZA,La isla
7,cota-vuelta_grande-1,COTA,Vuelta Grande
8,mosquera-balsillas-2,MOSQUERA,Balsillas
9,mosquera-balsillas-3,MOSQUERA,Balsillas


## Null summary

In [9]:
nulls = df.isna().sum()
nulls[nulls > 0]

Class                217
TipoCombustible      327
TipoFuenteEmision    326
Cuenca                76
dtype: int64

## Save

In [10]:
OUT_DIR.mkdir(exist_ok=True)
out_path = OUT_DIR / "industries.json"

records = df.astype(object).where(df.notna(), None).to_dict(orient="records")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=1)

print("Saved", len(records), "records to", out_path)

Saved 536 records to ..\Data\industries.json
